In [2]:
import os
import json
import pandas as pd

path = "../results"

files = os.listdir(path)

print(files)

['Hans-Tanja.json', 'Diana-Olga.json', 'Diana-Simon.json', 'Magnus-Simon.json', 'Hans-Magnus.json', 'Simon-Tanja.json', 'Magnus-Tanja.json', 'Diana-Tanja.json', 'Diana-Magnus.json', 'Magnus-Olga.json', 'Hans-Simon.json', 'Olga-Simon.json', 'Hans-Olga.json', 'Diana-Hans.json', 'Olga-Tanja.json']


In [3]:
folder = "../results"   

all_rows = []

files = os.listdir(folder)

for file in files:
    if ".json" in file:
        path = folder + "/" + file
        
        f = open(path)
        data = json.load(f)
        
        runs = data["runs"]
        
        for run in runs:
            events = run["events"]
            
            for event in events:
                row = {}
                
                row["run_id"] = run["run_id"]
                row["player_1"] = run["player_1"]
                row["player_2"] = run["player_2"]
                
                row["timestamp"] = event.get("timestamp")
                row["player"] = event.get("player")
                row["type"] = event.get("type")
                row["bar"] = event.get("bar")
                row["side"] = event.get("side")
                row["successful"] = event.get("successful")
                
                all_rows.append(row)

df = pd.DataFrame(all_rows)

print(df.shape)
print(df.columns)
print(df.head())

(29337, 9)
Index(['run_id', 'player_1', 'player_2', 'timestamp', 'player', 'type', 'bar',
       'side', 'successful'],
      dtype='object')
   run_id player_1 player_2  timestamp player     type       bar    side  \
0       0     Hans    Tanja   1.114063   Hans  contact   Middle1  Middle   
1       0     Hans    Tanja   3.088873   Hans  contact  Defense1    Left   
2       0     Hans    Tanja   6.050768   Hans  contact     Goal1  Middle   
3       0     Hans    Tanja   7.448406   Hans     shot      None    None   
4       0     Hans    Tanja   7.448406  Tanja  contact   Attack2  Middle   

  successful  
0       None  
1       None  
2       None  
3      False  
4       None  


In [4]:
rows = []

for f in files:
    if ".json" in f:
        file_path = folder + "/" + f
        
        with open(file_path, "r") as file:
            data = json.load(file)
        
        for run in data["runs"]:
            for event in run["events"]:
                
                row = {}
                row["player"] = event["player"]
                row["type"] = event["type"]
                row["time"] = event["timestamp"]
                row["bar"] = event.get("bar")
                row["side"] = event.get("side")
                row["success"] = event.get("successful")
                
                rows.append(row)

df = pd.DataFrame(rows)

print(len(df))
df.head()

29337


,player,type,time,bar,side,success
0,Hans,contact,1.114063,Middle1,Middle,None
1,Hans,contact,3.088873,Defense1,Left,None
2,Hans,contact,6.050768,Goal1,Middle,None
3,Hans,shot,7.448406,None,None,False
4,Tanja,contact,7.448406,Attack2,Middle,None


In [5]:
print(df["type"].value_counts())
print(df["player"].value_counts())

type
contact    22971
shot        6366
Name: count, dtype: int64
player
Olga      5969
Magnus    5230
Tanja     5100
Simon     5084
Hans      4051
Diana     3903
Name: count, dtype: int64


In [6]:
summary = df.groupby(["player", "type"]).size().unstack()

summary = summary.fillna(0)

shots = df[df["type"] == "shot"]

success = shots.groupby("player")["success"].sum()

summary["successful"] = success
summary["rate"] = summary["successful"] / summary["shot"]

summary = summary.fillna(0)

summary

/var/folders/wp/t1_mjxv966j18h0_x452h6_m0000gn/T/ipykernel_53325/1911555878.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  summary = summary.fillna(0)


type,contact,shot,successful,rate
player,,,,
Diana,2959,944,240,0.254237
Hans,2804,1247,162,0.129912
Magnus,4228,1002,186,0.185629
Olga,5305,664,223,0.335843
Simon,3716,1368,219,0.160088
Tanja,3959,1141,189,0.165644


In [7]:
side_counts = df.groupby(["player", "side"]).size().unstack()
side_counts = side_counts.fillna(0)
side_percent = side_counts.div(side_counts.sum(axis=1), axis=0)
side_percent

side,Left,Middle,Right
player,,,
Diana,0.375803,0.436972,0.187225
Hans,0.273181,0.473252,0.253566
Magnus,0.285005,0.455298,0.259697
Olga,0.294628,0.439397,0.265975
Simon,0.293864,0.435145,0.270990
Tanja,0.277090,0.447840,0.275069


In [8]:
shots = df[df["type"] == "shot"]

result = shots.groupby("player")["success"].value_counts().unstack().fillna(0)

result["success_rate"] = result[True] / (result[True] + result[False])

result

success,False,True,success_rate
player,,,
Diana,704,240,0.254237
Hans,1085,162,0.129912
Magnus,816,186,0.185629
Olga,441,223,0.335843
Simon,1149,219,0.160088
Tanja,952,189,0.165644


In [9]:
df2 = df.copy()

# make sure time is numeric
df2["time"] = pd.to_numeric(df2["time"], errors="coerce")

# sort by time
df2 = df2.sort_values("time").reset_index(drop=True)

# keep only unsuccessful shots
shots = df2[(df2["type"] == "shot") & (df2["success"] == False)].copy()

save_rows = []

for _, shot in shots.iterrows():
    shot_player = shot["player"]
    shot_time = shot["time"]

    # look at events happening at same time or within 1 second after shot
    next_events = df2[
        (df2["time"] >= shot_time) &
        (df2["time"] <= shot_time + 1) &
        (df2["player"] != shot_player)
    ].copy()

    # keep only defence contact events
    defence_events = next_events[
        (next_events["type"] == "contact") &
        (next_events["bar"].astype(str).str.contains("Defense", case=False, na=False))
    ]

    # if found, count first one as estimated successful defence
    if len(defence_events) > 0:
        first_def = defence_events.iloc[0]

        save_rows.append({
            "shot_player": shot_player,
            "shot_time": shot_time,
            "def_player": first_def["player"],
            "def_time": first_def["time"],
            "def_bar": first_def["bar"]
        })

# make dataframe
saves_df = pd.DataFrame(save_rows)

# count successful defences per player
if len(saves_df) > 0:
    defence_summary = saves_df.groupby("def_player").size().reset_index(name="successful_defences")
else:
    defence_summary = pd.DataFrame(columns=["def_player", "successful_defences"])

# count total defence contacts per player
all_def_contacts = df2[
    (df2["type"] == "contact") &
    (df2["bar"].astype(str).str.contains("Defense", case=False, na=False))
].groupby("player").size().reset_index(name="total_defence_contacts")

# merge both
defence_summary = defence_summary.merge(
    all_def_contacts,
    left_on="def_player",
    right_on="player",
    how="outer"
).fillna(0)

# clean columns
defence_summary["player_name"] = defence_summary["def_player"].fillna(defence_summary["player"])
defence_summary = defence_summary[["player_name", "successful_defences", "total_defence_contacts"]]

# defence rate
defence_summary["defence_rate"] = (
    defence_summary["successful_defences"] / defence_summary["total_defence_contacts"]
)

# sort
defence_summary = defence_summary.sort_values("successful_defences", ascending=False)

print("Defence performance summary:")
display(defence_summary)

print("estimated successful defences:")
display(saves_df.head(10))

Defence performance summary:


,player_name,successful_defences,total_defence_contacts,defence_rate
3,Olga,1164,1610,0.722981
4,Simon,939,1372,0.684402
2,Magnus,926,1170,0.791453
5,Tanja,798,1095,0.728767
0,Diana,644,727,0.885832
1,Hans,542,751,0.721704


estimated successful defences:


,shot_player,shot_time,def_player,def_time,def_bar
0,Hans,3.284543,Tanja,3.657221,Defense2
1,Diana,4.692704,Simon,4.754683,Defense2
2,Magnus,5.490359,Diana,5.672584,Defense1
3,Olga,5.851890,Magnus,5.890707,Defense2
4,Simon,6.169604,Olga,6.180025,Defense1
5,Olga,6.352071,Diana,6.617902,Defense1
6,Diana,6.462153,Simon,6.665521,Defense2
7,Olga,6.869492,Simon,6.869492,Defense2
8,Hans,7.045009,Magnus,7.242240,Defense2
9,Olga,7.138081,Magnus,7.242240,Defense2
